# Validation metrics

Compares synthetic draws with observed data using distributional, correlation, and error metrics. The notebook expects generated `.npz` samples, the observed model-ready table, and feature lists. It generates metric and correlation summaries.

In [ ]:
import ast
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import sys
import scipy.stats as stats
%matplotlib inline
import random
import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns', None)
random.seed(42)  # reproducibility
sys.path.append(r'..\src') 


path = '../'
CVAE_name = "best_cvae_model.pth"

In [ ]:
# Load column names
with open(path +"data/pref_cat.txt", "r") as f:
    cat_features = f.read().splitlines()
with open(path +"data/pref_num.txt", "r") as f:
    num_features = f.read().splitlines()
with open(path +"data/con_num.txt", "r") as f:
    num_cond_features = f.read().splitlines()
with open(path +"data/con_cat.txt", "r") as f:
    cat_cond_features = f.read().splitlines()

scaler_cond = torch.load(path + "models/lognorm/scaler_cond.pth", weights_only=False)

In [ ]:
df = pd.read_csv(path + 'data\TUmodeldata_full_cleaned.csv')

for year in df['DiaryYear'].unique():
    mask = df['DiaryYear'] == year
    total_weight = df.loc[mask, 'SessionWeight'].sum()
    lengthdf = len(df)/len(df['DiaryYear'].unique())
    df.loc[mask, 'SessionWeight'] = df.loc[mask, 'SessionWeight'] / total_weight * lengthdf

In [ ]:
# Load the model data - validation or training set
data_split = 'train'  # 'train' or 'val'
dataset = torch.load(path + f"results/datasets/{data_split}_dataset.pth", weights_only=False)

# Unpack the tensors directly
x_trips, x_lengths, x_purpose, x_num, x_cat, w = dataset[:]
dfin1 = pd.DataFrame(x_num, columns=num_cond_features)
dfin2 = pd.DataFrame(x_cat, columns=cat_cond_features)
dfin = pd.concat([dfin1, dfin2], axis=1)

In [ ]:
# Load the generated data
synthetic = np.load(path + f"results/val_samples_{CVAE_name.split('.')[0]}/synthetic_{data_split}.npz", allow_pickle=True)
synthetic = {key: synthetic[key] for key in synthetic.keys()}
print(synthetic.keys())  # dict_keys(['trips', 'lengths', 'purposes', 'mileage'])

x_mileage = x_trips * x_lengths  # Shape: (batch_size, num_modes)
synthetic['mileage'] = synthetic['trips'] * synthetic['lengths']

print("Trips shape:", synthetic["trips"].shape)        # (num_samples, batch, num_modes)
print("Lengths shape:", synthetic["lengths"].shape)    # (num_samples, batch, num_modes)
print("Purposes shape:", synthetic["purposes"].shape)  # (num_samples, batch)
print("\nExample sample:")
print("Trips:", synthetic["trips"][0, 0])
print("Lengths:", synthetic["lengths"][0, 0])
print("Purpose:", synthetic["purposes"][0, 0])

In [ ]:
modes = ['Walk', 'Bicycle', 'E-bike', 'Car', 'Car passenger', 'Public Transport', 'Moped']

def plot_mardist_comparison(x_trips, x_lengths, synthetic, confidence_level=0.95):
    plt.rcParams.update({'font.size': 8})

    if hasattr(x_trips, 'numpy'):
        original_trips = x_trips.numpy()
        original_lengths = x_lengths.numpy()
    else:
        original_trips = x_trips
        original_lengths = x_lengths

    generated_trips = np.array(synthetic["trips"])
    generated_lengths = np.array(synthetic["lengths"])

    num_samples, batch_size, num_modes = generated_trips.shape

    assert original_trips.shape[0] == batch_size, \
        f"Batch size mismatch: original {original_trips.shape[0]} vs generated {batch_size}"

    mode_labels = modes[:num_modes] if len(modes) >= num_modes else \
        modes + [f'Mode {i+1}' for i in range(len(modes), num_modes)]

    alpha = 1 - confidence_level
    lower_percentile = (alpha / 2) * 100
    upper_percentile = (1 - alpha / 2) * 100

    fig, axes = plt.subplots(num_modes, 2, figsize=(6, 16))

    if num_modes == 1:
        axes = axes.reshape(1, -1)

    # x-axis limits per mode
    xlim_map = {
        0: (0, 15),
        1: (0, 30),
        2: (0, 30),
        3: (0, 100),
        4: (0, 100),
        5: (0, 100),
        6: (0, 35),
    }

    for m in range(num_modes):
        orig_trips_mode   = original_trips[:, m]
        orig_lengths_mode = original_lengths[:, m]
        gen_trips_mode    = generated_trips[:, :, m]
        gen_lengths_mode  = generated_lengths[:, :, m]

        orig_trips_nonzero = orig_trips_mode[orig_trips_mode > 0]
        gen_trips_all      = gen_trips_mode.flatten()
        gen_trips_nonzero  = gen_trips_all[gen_trips_all > 0]

        if orig_trips_nonzero.size > 0 or gen_trips_nonzero.size > 0:
            unique_vals = np.union1d(
                np.unique(orig_trips_nonzero) if orig_trips_nonzero.size > 0 else [],
                np.unique(gen_trips_nonzero)  if gen_trips_nonzero.size  > 0 else []
            ).astype(int)

            original_counts = np.array([np.sum(orig_trips_nonzero == v) for v in unique_vals])

            gen_counts_per_sample = np.zeros((num_samples, len(unique_vals)))
            for s in range(num_samples):
                gen_s_nonzero = gen_trips_mode[s][gen_trips_mode[s] > 0]
                for i, v in enumerate(unique_vals):
                    gen_counts_per_sample[s, i] = np.sum(gen_s_nonzero == v)

            gen_mean  = np.mean(gen_counts_per_sample, axis=0)
            gen_lower = np.percentile(gen_counts_per_sample, lower_percentile, axis=0)
            gen_upper = np.percentile(gen_counts_per_sample, upper_percentile, axis=0)

            width = 0.35
            x_pos = np.arange(len(unique_vals))

            axes[m, 0].bar(x_pos - width/2, original_counts, width,
                           color='blue', alpha=0.7, label='Original')
            axes[m, 0].bar(x_pos + width/2, gen_mean, width,
                           color='orange', alpha=0.7, label='Generated')
            axes[m, 0].errorbar(x_pos + width/2, gen_mean,
                                yerr=[np.maximum(gen_mean - gen_lower, 0),
                                      np.maximum(gen_upper - gen_mean, 0)],
                                fmt='none', color='black', capsize=3, alpha=0.6,
                                label=f'{int(confidence_level*100)}% CI')

            axes[m, 0].set_xticks(x_pos)
            axes[m, 0].set_xticklabels(unique_vals)
            axes[m, 0].legend()

        axes[m, 0].set_title(f'{mode_labels[m]}')
        axes[m, 0].set_ylabel('Frequency')
        axes[m, 0].grid(alpha=0.3, axis='y')
        if m == num_modes - 1:
            axes[m, 0].set_xlabel('Trip Count')

        orig_mask    = orig_trips_mode > 0
        orig_len     = orig_lengths_mode[orig_mask]

        gen_len_list = []
        for s in range(num_samples):
            mask = gen_trips_mode[s] > 0
            gen_len_list.append(gen_lengths_mode[s][mask])

        xlim = xlim_map.get(m, (0, 100))
        x_range = np.linspace(xlim[0], xlim[1], 300)

        if orig_len.size > 1:
            kde_orig = stats.gaussian_kde(orig_len)
            axes[m, 1].plot(x_range, kde_orig(x_range),
                            color='blue', linewidth=2, label='Original (KDE)')

        kde_densities = []
        for gen_len in gen_len_list:
            if len(gen_len) > 1:
                kde = stats.gaussian_kde(gen_len)
                kde_densities.append(kde(x_range))

        if kde_densities:
            kde_arr   = np.array(kde_densities)
            kde_mean  = np.mean(kde_arr, axis=0)
            kde_lower = np.percentile(kde_arr, lower_percentile, axis=0)
            kde_upper = np.percentile(kde_arr, upper_percentile, axis=0)

            axes[m, 1].plot(x_range, kde_mean, color='darkorange',
                            linewidth=2, linestyle='--', label='Generated (KDE)')
            axes[m, 1].fill_between(x_range, kde_lower, kde_upper,
                                    color='red', alpha=0.2,
                                    label=f'{int(confidence_level*100)}% CI')

        axes[m, 1].set_xlim(xlim)
        axes[m, 1].set_title(f'{mode_labels[m]}')
        axes[m, 1].set_ylabel('Density')
        axes[m, 1].legend()
        axes[m, 1].grid(alpha=0.3, axis='y')
        if m == num_modes - 1:
            axes[m, 1].set_xlabel('Length')

    plt.tight_layout()
    plt.savefig(path + f"results/plots/trip_length_comparison_{data_split}.png", dpi=300)
    plt.show()
    plt.close()

    return fig, axes

In [ ]:
plot_mardist_comparison(x_trips, x_lengths, synthetic, confidence_level=0.95)

In [ ]:
def compute_metrics(x_trips, x_lengths, x_purpose, synthetic, confidence_level=0.95):
    """
    Compute validation metrics between original and generated data.
    
    Returns a DataFrame with per-mode metrics and scalar metrics for purpose and coverage.
    """
    from scipy import stats
    from scipy.spatial.distance import jensenshannon

    if hasattr(x_trips, 'numpy'):
        original_trips   = x_trips.numpy()
        original_lengths = x_lengths.numpy()
        original_purpose = x_purpose.numpy()
    else:
        original_trips   = np.array(x_trips)
        original_lengths = np.array(x_lengths)
        original_purpose = np.array(x_purpose)

    generated_trips   = np.array(synthetic["trips"])    # (num_samples, batch, num_modes)
    generated_lengths = np.array(synthetic["lengths"])
    num_samples, batch_size, num_modes = generated_trips.shape

    alpha = 1 - confidence_level
    lower_p = (alpha / 2) * 100
    upper_p = (1 - alpha / 2) * 100

    rows = []
    for m in range(num_modes):
        orig_t = original_trips[:, m]
        orig_l = original_lengths[:, m]
        gen_t  = generated_trips[:, :, m]   # (num_samples, batch)
        gen_l  = generated_lengths[:, :, m]

        # Trips 
        trips_mae  = np.abs(orig_t - gen_t_mean).mean()

        # JSD on trip count distributions (non-zero only)
        all_vals = np.union1d(np.unique(orig_t), np.unique(gen_t.flatten())).astype(int)
        orig_counts = np.array([np.sum(orig_t == v) for v in all_vals], dtype=float)
        gen_counts  = np.array([np.sum(gen_t.flatten() == v) for v in all_vals], dtype=float)
        orig_counts /= orig_counts.sum()
        gen_counts  /= gen_counts.sum()
        trips_jsd = jensenshannon(orig_counts, gen_counts)

        # Lengths (non-zero only) 
        orig_l_nz = orig_l[orig_t > 0]
        gen_l_nz  = gen_l.flatten()[gen_t.flatten() > 0]

        if orig_l_nz.size > 1 and gen_l_nz.size > 1:
            lengths_mae = np.abs(orig_l_nz.mean() - gen_l_nz.mean())
            lengths_wd  = stats.wasserstein_distance(orig_l_nz, gen_l_nz)
            lengths_ks  = stats.ks_2samp(orig_l_nz, gen_l_nz).statistic
        else:
            lengths_mae = lengths_wd = lengths_ks = np.nan

        # Coverage
        gen_l_lower = np.percentile(gen_l, lower_p, axis=0)
        gen_l_upper = np.percentile(gen_l, upper_p, axis=0)
        coverage = np.mean((orig_l >= gen_l_lower) & (orig_l <= gen_l_upper))

        rows.append({
            'mode':       m + 1,
            'trips_mae':  round(trips_mae, 4),
            'trips_jsd':  round(trips_jsd, 4),
            'lengths_mae': round(lengths_mae, 4),
            'lengths_ks':  round(lengths_ks, 4),
            'lengths_wd':  round(lengths_wd, 4),
            'coverage':    round(coverage, 4),
        })

    metrics_df = pd.DataFrame(rows).set_index('mode')

    # Purpose JSD
    gen_purpose = np.array(synthetic["purposes"]).flatten()
    all_purp    = np.union1d(np.unique(original_purpose), np.unique(gen_purpose)).astype(int)
    orig_p = np.array([np.sum(original_purpose == v) for v in all_purp], dtype=float)
    gen_p  = np.array([np.sum(gen_purpose == v)      for v in all_purp], dtype=float)
    orig_p /= orig_p.sum()
    gen_p  /= gen_p.sum()
    purpose_jsd = round(jensenshannon(orig_p, gen_p), 4)

    print(metrics_df.to_string())
    print(f"\nPurpose JSD: {purpose_jsd}")

    return metrics_df, purpose_jsd

In [ ]:
metrics_df, purpose_jsd = compute_metrics(x_trips, x_lengths, x_purpose, synthetic)
metrics_df.to_csv(path + f"results/val_samples_best_cvae_model/metrics_{data_split}.csv")

In [ ]:
def plot_trip_length_correlation(x_trips, x_lengths, synthetic):
    """
    Compare Spearman correlations between trip counts and average lengths
    across modes â€” original vs generated.
    """
    from scipy import stats

    if hasattr(x_trips, 'numpy'):
        original_trips   = x_trips.numpy()
        original_lengths = x_lengths.numpy()
    else:
        original_trips   = np.array(x_trips)
        original_lengths = np.array(x_lengths)

    generated_trips   = np.array(synthetic["trips"])    # (num_samples, batch, num_modes)
    generated_lengths = np.array(synthetic["lengths"])
    num_samples, batch_size, num_modes = generated_trips.shape

    # Compute correlations
    orig_corrs = []
    gen_corrs  = []
    gen_corrs_lower = []
    gen_corrs_upper = []

    for m in range(num_modes):
        orig_t = original_trips[:, m]
        orig_l = original_lengths[:, m]

        # Original: non-zero only
        mask   = orig_t > 0
        rho, _ = stats.spearmanr(orig_t[mask], orig_l[mask]) if mask.sum() > 2 else (np.nan, np.nan)
        orig_corrs.append(rho)

        # Generated: one correlation per sample, then summarise
        sample_rhos = []
        for s in range(num_samples):
            gen_t = generated_trips[s, :, m]
            gen_l = generated_lengths[s, :, m]
            smask = gen_t > 0
            if smask.sum() > 2:
                r, _ = stats.spearmanr(gen_t[smask], gen_l[smask])
                sample_rhos.append(r)

        if sample_rhos:
            gen_corrs.append(np.mean(sample_rhos))
            gen_corrs_lower.append(np.percentile(sample_rhos, 2.5))
            gen_corrs_upper.append(np.percentile(sample_rhos, 97.5))
        else:
            gen_corrs.append(np.nan)
            gen_corrs_lower.append(np.nan)
            gen_corrs_upper.append(np.nan)

    # Print summary table 
    mode_names = ['Walk', 'Bicycle', 'E-bike', 'Car', 'Car passenger', 'PT', 'Moped']
    corr_df = pd.DataFrame({
        'mode':           mode_names[:num_modes],
        'orig_spearman':  np.round(orig_corrs, 4),
        'gen_spearman':   np.round(gen_corrs,  4),
        'gen_lower_95':   np.round(gen_corrs_lower, 4),
        'gen_upper_95':   np.round(gen_corrs_upper, 4),
    }).set_index('mode')
    print(corr_df.to_string())

    fig, ax = plt.subplots(figsize=(8, 4))
    x_pos = np.arange(num_modes)
    width = 0.35

    ax.bar(x_pos - width/2, orig_corrs, width,
           color='blue', alpha=0.7, label='Original')
    ax.bar(x_pos + width/2, gen_corrs, width,
           color='orange', alpha=0.7, label='Generated')

    # CI error bars on generated
    yerr_lower = np.maximum(np.array(gen_corrs) - np.array(gen_corrs_lower), 0)
    yerr_upper = np.maximum(np.array(gen_corrs_upper) - np.array(gen_corrs), 0)
    ax.errorbar(x_pos + width/2, gen_corrs,
                yerr=[yerr_lower, yerr_upper],
                fmt='none', color='black', capsize=4, alpha=0.7, label='95% CI')

    ax.set_xticks(x_pos)
    ax.set_xticklabels(mode_names[:num_modes], rotation=15, ha='right')
    ax.set_ylabel("Spearman's Ï")
    ax.set_title('Trip count vs. average length correlation')
    ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
    ax.legend()
    ax.grid(alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig(path + f"results/plots/correlation_comparison_{data_split}.png", dpi=300)
    plt.show()
    plt.close()

    return corr_df

In [ ]:
corr_df = plot_trip_length_correlation(x_trips, x_lengths, synthetic)
corr_df.to_csv(path + f"results/val_samples_best_cvae_model/correlations_{data_split}.csv")

## Temporal validation for each mode

In [ ]:
def discretize_variables(
    df, 
    columns=None, 
    method="quantile", 
    bins=4, 
    labels=None, 
    custom_bins=None
):
    """
    Discretize (bin) one or multiple numerical variables into categorical bins.
    This version replaces the original columns with their binned versions.

    Parameters:
        df : pd.DataFrame
            Input DataFrame.
        columns : list or None
            Columns to discretize. If None, all numeric columns are used.
        method : str, default="quantile"
            Discretization method:
            - "quantile" â†’ equal-frequency binning
            - "uniform" â†’ equal-width binning
            - "custom" â†’ user-provided bin edges
        bins : int, default=4
            Number of bins (ignored if method="custom").
        labels : list or None
            Labels for the bins. If None, interval ranges are used.
        custom_bins : dict or list or None
            - If dict â†’ {"col1": [...], "col2": [...]} custom bins per column
            - If list â†’ same custom bins for all columns
            - Required if method="custom".

    Returns:
        df_new : pd.DataFrame
            DataFrame with original columns replaced by binned versions.
    """
    df_new = df.copy()

    # Default: all numeric columns
    if columns is None:
        columns = df.select_dtypes(include=[np.number]).columns.tolist()

    for col in columns:
        if method == "quantile":
            df_new[col] = pd.qcut(df[col], q=bins, labels=labels, duplicates="drop")
        elif method == "uniform":
            df_new[col] = pd.cut(df[col], bins=bins, labels=labels)
        elif method == "custom":
            if custom_bins is None:
                raise ValueError("You must provide 'custom_bins' when method='custom'.")
            if isinstance(custom_bins, dict):
                bins_for_col = custom_bins.get(col)
                if bins_for_col is None:
                    raise ValueError(f"No custom bins provided for column {col}")
            else:
                bins_for_col = custom_bins
            df_new[col] = pd.cut(df[col], bins=bins_for_col, labels=labels)
        else:
            raise ValueError("Invalid method. Choose from 'quantile', 'uniform', or 'custom'.")
    
    return df_new

def calculate_sum_metrics_with_ci(df_orig, x_trips, x_mileage, x_purposes, synthetic, comb, 
                                  value_type='trips', confidence_level=0.95):
    """
    Calculate distribution similarity metrics (RÂ², SRMSE, Pearson r) with confidence intervals
    for trips, mileage, or purposes grouped by conditional features.
    
    Parameters:
        df_orig: original DataFrame with conditional features
        x_trips: original trips tensor/array (batch_size, num_modes)
        x_mileage: original mileage tensor/array (batch_size, num_modes)
        x_purposes: original purposes tensor/array (batch_size,)
        synthetic: dict with 'trips', 'mileage', and 'purposes' arrays
                   - trips/mileage: (num_samples, batch_size, num_modes)
                   - purposes: (num_samples, batch_size)
        comb: list of columns from df_orig to group by
        value_type: 'trips', 'mileage', or 'purposes'
        confidence_level: confidence level for intervals (default: 0.95)
    
    Returns:
        merged: DataFrame with aggregated values and confidence intervals
        metrics: dict with mean metrics and their confidence intervals
    """
    from scipy.stats import pearsonr
    from sklearn.metrics import r2_score
    
    # Convert to numpy if needed
    if hasattr(x_trips, 'numpy'):
        x_trips = x_trips.numpy()
        x_mileage = x_mileage.numpy()
        x_purposes = x_purposes.numpy()
    
    # Get synthetic data
    synthetic_trips = np.array(synthetic["trips"])  # (num_samples, batch_size, num_modes)
    synthetic_mileage = np.array(synthetic["mileage"])  # (num_samples, batch_size, num_modes)
    synthetic_purposes = np.array(synthetic["purposes"])  # (num_samples, batch_size)
    
    num_samples = synthetic_trips.shape[0]
    batch_size = synthetic_trips.shape[1]
    
    # Verify batch size matches
    assert len(df_orig) == batch_size, \
        f"Batch size mismatch: df_orig {len(df_orig)} vs data {batch_size}"
    
    # Calculate percentiles for confidence intervals
    alpha = 1 - confidence_level
    lower_percentile = (alpha / 2) * 100
    upper_percentile = (1 - alpha / 2) * 100
    
    # Handle purposes differently (single column, categorical)
    if value_type == 'purposes':
        # Original purposes
        df_orig_full = df_orig[comb].copy()
        df_orig_full['purpose'] = x_purposes
        
        # Count purposes per combination
        orig_counts = df_orig_full.groupby(comb + ['purpose'], observed=False).size().reset_index(name='count_orig')
        
        # Process each generated sample
        all_gen_counts = []
        all_metrics = {'r2': [], 'rmse': [], 'r': []}
        
        for sample_idx in range(num_samples):
            df_gen_sample = df_orig[comb].copy()
            df_gen_sample['purpose'] = synthetic_purposes[sample_idx]
            
            # Count purposes per combination
            gen_counts = df_gen_sample.groupby(comb + ['purpose'], observed=False).size().reset_index(name='count_gen')
            
            # Merge with original counts to ensure same rows and order
            merged_sample = pd.merge(orig_counts, gen_counts, on=comb + ['purpose'], how='left')
            merged_sample['count_gen'] = merged_sample['count_gen'].fillna(0).astype(float)
            
            all_gen_counts.append(merged_sample)
            
            # Compute metrics safely
            prob_orig = merged_sample['count_orig'].astype(float)
            prob_gen = merged_sample['count_gen'].astype(float)
            
            r2 = r2_score(prob_orig, prob_gen)
            mse = np.mean((prob_orig - prob_gen) ** 2)
            rmse = np.sqrt(mse)
            r, _ = pearsonr(prob_orig, prob_gen) if len(prob_orig) > 1 else (np.nan, np.nan)
            
            all_metrics['r2'].append(r2)
            all_metrics['rmse'].append(rmse)
            all_metrics['r'].append(r)
        
        # Aggregate across samples
        gen_counts_stacked = np.stack([df_counts['count_gen'].values for df_counts in all_gen_counts])
        
        # Use first sample's structure
        df_final = all_gen_counts[0][comb + ['purpose', 'count_orig']].copy()
        df_final = df_final.rename(columns={'count_orig': 'orig'})
        df_final['gen'] = np.mean(gen_counts_stacked, axis=0)
        df_final['gen_lower'] = np.percentile(gen_counts_stacked, lower_percentile, axis=0)
        df_final['gen_upper'] = np.percentile(gen_counts_stacked, upper_percentile, axis=0)
        
    else:
        # Handle trips or mileage (multiple modes)
        if value_type == 'trips':
            orig_values = x_trips
            gen_values = synthetic_trips
        else:  # 'mileage'
            orig_values = x_mileage
            gen_values = synthetic_mileage
        
        num_modes = orig_values.shape[1]
        mode_cols = [f'mode_{i+1}' for i in range(num_modes)]
        
        # Build original dataframe
        df_orig_full = df_orig[comb].copy()
        for i, col in enumerate(mode_cols):
            df_orig_full[col] = orig_values[:, i]
        
        # Aggregate original by combination
        orig_sum = df_orig_full.groupby(comb, observed=False)[mode_cols].sum().reset_index()
        orig_sum = orig_sum.rename(columns={col: f"{col}_orig" for col in mode_cols})
        
        # Process each generated sample
        all_gen_sums = []
        all_metrics = {'r2': [], 'rmse': [], 'r': []}
        
        for sample_idx in range(num_samples):
            # Build generated dataframe for this sample
            df_gen_sample = df_orig[comb].copy()
            for i, col in enumerate(mode_cols):
                df_gen_sample[col] = gen_values[sample_idx, :, i]
            
            # Aggregate by combination
            gen_sum = df_gen_sample.groupby(comb, observed=False)[mode_cols].sum().reset_index()
            gen_sum = gen_sum.rename(columns={col: f"{col}_gen" for col in mode_cols})
            
            # Merge with original
            merged_sample = pd.merge(orig_sum, gen_sum, on=comb, how='outer')
            
            # Melt to long format
            value_cols_orig = [f"{col}_orig" for col in mode_cols]
            value_cols_gen = [f"{col}_gen" for col in mode_cols]
            
            df_long = pd.melt(
                merged_sample,
                id_vars=comb,
                value_vars=value_cols_orig + value_cols_gen,
                var_name="mode_type",
                value_name="value"
            )
            
            # Split mode and type
            df_long[['mode', 'type']] = df_long['mode_type'].str.rsplit('_', n=1, expand=True)
            
            # Pivot to have orig and gen in same row
            df_pivot = df_long.pivot_table(
                index=comb + ['mode'],
                columns='type',
                values='value',
                fill_value=0
            ).reset_index()
            
            all_gen_sums.append(df_pivot)
            
            # Calculate metrics for this sample
            prob_orig = df_pivot['orig'].fillna(0).astype(float)
            prob_gen = df_pivot['gen'].fillna(0).astype(float)
            
            r2 = r2_score(prob_orig, prob_gen)
            mse = np.mean((prob_orig - prob_gen) ** 2)
            rmse = np.sqrt(mse)
            r, _ = pearsonr(prob_orig, prob_gen) if len(prob_orig) > 1 else (np.nan, np.nan)
            
            all_metrics['r2'].append(r2)
            all_metrics['rmse'].append(rmse)
            all_metrics['r'].append(r)
        
        # Aggregate across samples
        gen_values_stacked = []
        for df_pivot in all_gen_sums:
            gen_values_stacked.append(df_pivot['gen'].values)
        
        gen_values_stacked = np.array(gen_values_stacked)
        
        # Use first sample's structure
        df_final = all_gen_sums[0][comb + ['mode', 'orig']].copy()
        df_final['gen'] = np.mean(gen_values_stacked, axis=0)
        df_final['gen_std'] = np.std(gen_values_stacked, axis=0)
        df_final['gen_lower'] = np.percentile(gen_values_stacked, lower_percentile, axis=0)
        df_final['gen_upper'] = np.percentile(gen_values_stacked, upper_percentile, axis=0)
    
    # Calculate mean and CI for metrics
    metrics = {
        'r2_mean': np.mean(all_metrics['r2']),
        'r2_std': np.std(all_metrics['r2']),
        'r2_lower': np.percentile(all_metrics['r2'], lower_percentile),
        'r2_upper': np.percentile(all_metrics['r2'], upper_percentile),

        'rmse_mean': np.mean(all_metrics['rmse']),
        'rmse_std': np.std(all_metrics['rmse']),
        'rmse_lower': np.percentile(all_metrics['rmse'], lower_percentile),
        'rmse_upper': np.percentile(all_metrics['rmse'], upper_percentile),

        'r_mean': np.mean(all_metrics['r']),
        'r_std': np.std(all_metrics['r']),
        'r_lower': np.percentile(all_metrics['r'], lower_percentile),
        'r_upper': np.percentile(all_metrics['r'], upper_percentile),

        'n_samples': num_samples,
        'n_points': len(df_final)
    }
    
    return df_final, metrics

def plot_results(results_dict, value_type_label):
    """
    Plot results for a specific value type with simple subplot titles.

    Parameters:
    -----------
    results_dict : dict
        Dictionary containing results for each value type.
    value_type_label : str
        Key in results_dict to plot ('trips', 'mileage', 'purposes').
    """
    results = results_dict[value_type_label]
    n = len(results)
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3 * rows))
    axes = axes.flatten()
    
    for i, (merged, metrics) in enumerate(results):
        ax = axes[i]
        
        # Scatter plot with error bars
        ax.errorbar(
            merged['orig'], 
            merged['gen'], 
            yerr = [
                    np.maximum(merged['gen'] - merged['gen_lower'], 0),
                    np.maximum(merged['gen_upper'] - merged['gen'], 0)
                ],
            fmt='o', 
            alpha=0.6, 
            ecolor='lightgray', 
            capsize=3,
            markersize=5,
            elinewidth=1
        )
        
        # Perfect prediction line
        max_val = max(merged['orig'].max(), merged['gen_upper'].max())
        min_val = min(merged['orig'].min(), merged['gen_lower'].min())
        ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
        
        ax.set_xlabel("Original", fontsize=10)
        ax.set_ylabel("Generated (mean Â± 95% CI)", fontsize=10)
        
        # Simple subplot title
        ax.set_title(f"({i+1})", fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)
    
    # Remove extra axes if any
    for j in range(len(results), len(axes)):
        fig.delaxes(axes[j])
    
    plt.tight_layout()
    # plt.suptitle(
    #     f"{value_type_label.capitalize()} Distribution: Original vs. Generated (with 95% CI)", 
    #     fontsize=14, y=1.02
    # )
    
    plt.savefig(path + f"results/plots/{value_type_label}_comparison_{data_split}.png", dpi=300)
    plt.show()
    plt.close()

### Trips

In [ ]:
dfin_disc = discretize_variables(dfin, columns=num_cond_features[:-2], method="quantile", bins=12)

In [ ]:
year = [['DiaryYear']]

for value_type in ['trips']:
    
    for idx, comb in enumerate(year):
        merged, metrics = calculate_sum_metrics_with_ci(
            dfin_disc, x_trips, x_mileage, x_purpose, synthetic, comb, value_type=value_type
        )

import itertools
from sklearn.metrics import mean_squared_error, r2_score

agg_df = merged.groupby(["DiaryYear", "mode", ])[["gen", "orig"]].sum().reset_index()
modes = agg_df["mode"].unique()
colors = itertools.cycle(plt.cm.tab10.colors)  # 10 distinct colors

In [ ]:
year = [['DiaryYear']]

for value_type in ['trips']:
    
    for idx, comb in enumerate(year):
        merged, metrics = calculate_sum_metrics_with_ci(
            dfin_disc, x_trips, x_mileage, x_purpose, synthetic, comb, value_type=value_type
        )

import itertools
from sklearn.metrics import mean_squared_error, r2_score

agg_df = merged.groupby(["DiaryYear", "mode", ])[["gen", "orig"]].sum().reset_index()
modes = agg_df["mode"].unique()
colors = itertools.cycle(plt.cm.tab10.colors)  # 10 distinct colors

results = []

for mode in modes:
    mode_data = agg_df[agg_df["mode"] == mode].sort_values("DiaryYear")

    obs = mode_data["orig"].values
    gen = mode_data["gen"].values

    # --- correlation ---
    if np.std(obs) > 0 and np.std(gen) > 0:
        corr = np.corrcoef(obs, gen)[0, 1]
    else:
        corr = np.nan

    # --- RMSE ---
    rmse = np.sqrt(mean_squared_error(obs, gen))
    rmse_norm = rmse / np.mean(obs)

    # --- RÂ² ---
    r2 = r2_score(obs, gen)

    results.append({
        "mode": mode,
        "trend_r2": round(r2, 2),
        "trend_corr": round(corr, 2),
        "trend_rmse": round(rmse_norm, 2),
    })

trend_df = pd.DataFrame(results)
print(trend_df)

In [ ]:
plt.figure(figsize=(14, 8))

for mode in modes:
    mode_data = agg_df[agg_df["mode"] == mode].set_index("DiaryYear")
    color = next(colors)
    
    # gen (solid)
    plt.plot(mode_data.index, mode_data["gen"], marker="o", linestyle="-", color=color,
             label=f"{mode} - gen")
    
    # orig (dashed)
    plt.plot(mode_data.index, mode_data["orig"], marker="x", linestyle="--", color=color,
             label=f"{mode} - orig")

plt.title("Trends of Trips by Mode (Gen vs Orig, aggregated over genders)")
plt.xlabel("Year")
plt.ylabel("Mileage")
# plt.ylim(0,50000)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

### Mileage

In [ ]:
year = [['DiaryYear']]

for value_type in ['mileage']:
    
    for idx, comb in enumerate(year):
        merged, metrics = calculate_sum_metrics_with_ci(
            dfin_disc, x_trips, x_mileage, x_purpose, synthetic, comb, value_type=value_type
        )

In [ ]:
agg_df = merged.groupby(["DiaryYear", "mode", ])[["gen", "orig"]].sum().reset_index()
modes = agg_df["mode"].unique()
colors = itertools.cycle(plt.cm.tab10.colors)  # 10 distinct colors

results = []

for mode in modes:
    mode_data = agg_df[agg_df["mode"] == mode].sort_values("DiaryYear")

    obs = mode_data["orig"].values
    gen = mode_data["gen"].values

    # --- correlation ---
    if np.std(obs) > 0 and np.std(gen) > 0:
        corr = np.corrcoef(obs, gen)[0, 1]
    else:
        corr = np.nan

    # --- RMSE ---
    rmse = np.sqrt(mean_squared_error(obs, gen))
    rmse_norm = rmse / np.mean(obs)

    # --- RÂ² ---
    r2 = r2_score(obs, gen)

    results.append({
        "mode": mode,
        "trend_r2": round(r2, 2),
        "trend_corr": round(corr, 2),
        "trend_rmse": round(rmse_norm, 2),
    })

trend_df = pd.DataFrame(results)
print(trend_df)

In [ ]:
plt.figure(figsize=(14, 8))

for mode in modes:
    mode_data = agg_df[agg_df["mode"] == mode].set_index("DiaryYear")
    color = next(colors)
    
    # gen (solid)
    plt.plot(mode_data.index, mode_data["gen"], marker="o", linestyle="-", color=color,
             label=f"{mode} - gen")
    
    # orig (dashed)
    plt.plot(mode_data.index, mode_data["orig"], marker="x", linestyle="--", color=color,
             label=f"{mode} - orig")

plt.title("Trends of Mileage by Mode (Gen vs Orig, aggregated over genders)")
plt.xlabel("Year")
plt.ylabel("Mileage")
# plt.ylim(0,50000)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

## Calculate Conditional distributions metrics

In [ ]:
def calculate_metrics_per_mode(
    df_orig,
    x_trips,
    x_mileage,
    synthetic,
    comb,
    rare_quantile=0.33,
    tail_quantile=0.33,
):

    from scipy.stats import pearsonr
    from sklearn.metrics import r2_score

    # ----------------------------- #
    # Convert tensors to numpy
    # ----------------------------- #
    if hasattr(x_trips, 'numpy'):
        x_trips = x_trips.numpy()
        x_mileage = x_mileage.numpy()

    synthetic_trips = np.array(synthetic["trips"])
    synthetic_mileage = np.array(synthetic["mileage"])

    num_samples = synthetic_trips.shape[0]
    num_modes = x_trips.shape[1]
    mode_names = [f'mode_{i+1}' for i in range(num_modes)]

    value_sources = {
        "trips": (x_trips, synthetic_trips),
        "mileage": (x_mileage, synthetic_mileage),
    }

    # ================================================================ #
    # HELPER: distribution metrics (R2, corr, NRMSE) for one value type
    # ================================================================ #
    def distribution_metrics(orig_vals, gen_vals, mode_idx):
        df_obs_val = df_orig[comb].copy()
        df_obs_val["value"] = orig_vals[:, mode_idx]

        orig_sum = (
            df_obs_val.groupby(comb, observed=False)["value"]
            .sum().reset_index().rename(columns={"value": "orig"})
        )

        r2_list, corr_list, nrmse_list = [], [], []

        for s in range(num_samples):
            df_gen = df_orig[comb].copy()
            df_gen["value"] = gen_vals[s, :, mode_idx]

            gen_sum = (
                df_gen.groupby(comb, observed=False)["value"]
                .sum().reset_index().rename(columns={"value": "gen"})
            )

            merged = orig_sum.merge(gen_sum, on=comb, how="left")
            merged["gen"] = merged["gen"].fillna(0)

            o = merged["orig"].values.astype(float)
            g = merged["gen"].values.astype(float)

            r2_list.append(r2_score(o, g))

            if np.std(o) == 0 or np.std(g) == 0:
                corr_list.append(np.nan)
            else:
                corr_list.append(pearsonr(o, g)[0])

            rmse = np.sqrt(np.mean((o - g) ** 2))
            mean_o = np.mean(o)
            nrmse_list.append(rmse / mean_o if mean_o > 0 else np.nan)

        def clean(arr):
            arr = np.array(arr, dtype=float)
            return arr[~np.isnan(arr)]

        r2_list = clean(r2_list)
        corr_list = clean(corr_list)
        nrmse_list = clean(nrmse_list)

        return {
            "r2_mean": np.mean(r2_list) if len(r2_list) else np.nan,
            "corr_mean": np.mean(corr_list) if len(corr_list) else np.nan,
            "nrmse_mean": np.mean(nrmse_list) if len(nrmse_list) else np.nan,
        }

    # ================================================================ #
    # HELPER: Brier skill score (overall / rare / tail), subset baseline
    # ================================================================ #
    def brier_skill_metrics(mode_idx, rare_quantile=0.33, tail_quantile=0.33):
        y_true = (x_trips[:, mode_idx] > 0).astype(float)

        df_obs = df_orig[comb].copy()
        df_obs["adopt"] = y_true

        obs_group = (
            df_obs.groupby(comb, observed=False)["adopt"]
            .mean().reset_index().rename(columns={"adopt": "p_obs"})
        )
        cohort_sizes = (
            df_obs.groupby(comb, observed=False).size().reset_index(name="n")
        )

        # mean predicted adoption probability across samples
        y_pred = np.mean(synthetic_trips[:, :, mode_idx] > 0, axis=0)

        df_pred = df_orig[comb].copy()
        df_pred["p_pred"] = y_pred

        pred_group = (
            df_pred.groupby(comb, observed=False)["p_pred"]
            .mean().reset_index()
        )

        merged = (
            obs_group.merge(pred_group, on=comb).merge(cohort_sizes, on=comb)
        ).dropna()

        if len(merged) == 0:
            return {
                "brier_skill": np.nan,
                "brier_skill_rare": np.nan,
                "brier_skill_tail": np.nan,
            }

        # global mean used as the "naive" baseline prediction everywhere
        global_mean = np.average(merged["p_obs"], weights=merged["n"])

        def skill_for(subset):
            if len(subset) == 0:
                return np.nan

            brier = np.average(
                (subset["p_obs"] - subset["p_pred"]) ** 2, weights=subset["n"]
            )
            brier_baseline = np.average(
                (subset["p_obs"] - global_mean) ** 2, weights=subset["n"]
            )

            if brier_baseline <= 0:
                return np.nan

            return 1 - (brier / brier_baseline)


        brier_skill = skill_for(merged)

        # ---- RARE: bottom 33% of cohorts by sample size ---- #
        rare_cutoff = merged["n"].quantile(rare_quantile)
        rare_mask = merged["n"] <= rare_cutoff
        brier_skill_rare = skill_for(merged.loc[rare_mask])

        # ---- TAIL: bottom 33% of cohorts by observed adoption prob ---- #
        tail_cutoff = merged["p_obs"].quantile(tail_quantile)
        tail_mask = merged["p_obs"] <= tail_cutoff
        brier_skill_tail = skill_for(merged.loc[tail_mask])

        return {
            "brier_skill": brier_skill,
            "brier_skill_rare": brier_skill_rare,
            "brier_skill_tail": brier_skill_tail,
        }
    # ================================================================ #
    # LOOP OVER MODES
    # ================================================================ #
    mode_metrics = {}

    for mode_idx, mode_name in enumerate(mode_names):
        metrics = {}

        for value_type, (orig_vals, gen_vals) in value_sources.items():
            dist = distribution_metrics(orig_vals, gen_vals, mode_idx)
            for k, v in dist.items():
                metrics[f"{value_type}_{k}"] = v

        metrics.update(brier_skill_metrics(mode_idx, rare_quantile=rare_quantile, tail_quantile=tail_quantile))

        mode_metrics[mode_name] = metrics

    return mode_metrics

In [ ]:
import itertools

# Base pool (exclude HomeAdrGMMzoneL2)
base_cols = [col for col in cat_cond_features if col != 'HomeAdrGMMzoneL2'] + num_cond_features

# âœ… All unique pairs
pair_combs = list(itertools.combinations(base_cols, 2))

# âœ… Add zone as standalone
zone_combs = [('HomeAdrGMMzoneL2',)]

# âœ… Final combinations
combs = [list(c) for c in pair_combs] + [list(c) for c in zone_combs]

print(f"Total combinations: {len(combs)}")

In [ ]:
metrics_all = {}

for idx, comb in enumerate(combs):
    print(f"Calculating metrics for combination {idx+1}/{len(combs)}: {comb}")
    metrics = calculate_metrics_per_mode(
        dfin_disc, x_trips, x_mileage, synthetic, comb
    )

    # Save all metrics
    metrics_df = pd.DataFrame(metrics).T.reset_index().rename(columns={"index": "mode"})
    metrics_all[f"comb_{idx+1}"] = metrics_df

metrics_all_df = pd.concat(metrics_all, names=["combination"]).reset_index(level=0)

In [ ]:
summary_df = (
    metrics_all_df
    .groupby("mode")
    .agg({
        "trips_r2_mean": ["mean", "std"],
        "trips_corr_mean": ["mean", "std"],
        "trips_nrmse_mean": ["mean", "std"],
        "mileage_r2_mean": ["mean", "std"],
        "mileage_corr_mean": ["mean", "std"],
        "mileage_nrmse_mean": ["mean", "std"], 
        "brier_skill": ["mean", "std"],
        "brier_skill_rare": ["mean", "std"],
        "brier_skill_tail": ["mean", "std"],
    })
)

# flatten column names
summary_df.columns = ["_".join(col) for col in summary_df.columns]
summary_df = summary_df.reset_index()
summary_df = summary_df.round(2)
# Save the summary to CSV
summary_df.to_csv(path + f"results/metrics_summary_{data_split}.csv", index=False)

In [ ]:
# ============================================================ #
# CORE: compute raw (unbinned) cohort-level calibration data
# ============================================================ #
def compute_cohort_calibration_data(
    df_orig, x_trips, synthetic, comb, mode_idx,
    rare_quantile=0.33, tail_quantile=0.33
):
    if hasattr(x_trips, "numpy"):
        x_trips = x_trips.numpy()

    synthetic_trips = np.array(synthetic["trips"])

    values = x_trips[:, mode_idx]
    y_true = (values > 0).astype(float)
    y_pred = np.mean(synthetic_trips[:, :, mode_idx] > 0, axis=0)

    df = df_orig[comb].copy()
    df["p_obs"] = y_true
    df["p_pred"] = y_pred

    grouped = (
        df.groupby(comb, observed=False)
        .agg(p_obs=("p_obs", "mean"),
             p_pred=("p_pred", "mean"),
             n=("p_obs", "size"))
        .reset_index()
    )
    grouped = grouped[grouped["n"] > 0].copy()

    # flag rare / tail cohorts using the same quantile rule as the metrics
    rare_cutoff = grouped["n"].quantile(rare_quantile)
    tail_cutoff = grouped["p_obs"].quantile(tail_quantile)

    grouped["is_rare"] = grouped["n"] <= rare_cutoff
    grouped["is_tail"] = grouped["p_obs"] <= tail_cutoff

    def flag(row):
        if row["is_rare"] and row["is_tail"]:
            return "rare & tail"
        elif row["is_rare"]:
            return "rare"
        elif row["is_tail"]:
            return "tail"
        return "normal"

    grouped["group"] = grouped.apply(flag, axis=1)

    return grouped

# ============================================================ #
# PLOT: raw cohort scatter, color-coded by rare/tail membership
# ============================================================ #
def plot_cohort_calibration(calib, ax=None, title=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))

    if len(calib) == 0:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        if title:
            ax.set_title(title)
        return ax

    max_val = max(calib["p_pred"].max(), calib["p_obs"].max(), 0.001) * 1.1

    ax.plot([0, max_val], [0, max_val], '--', color='gray', zorder=1)

    colors = {
        "normal": "#4C72B0",
        "rare": "#DD8452",
        "tail": "#55A868",
        "rare & tail": "#C44E52",
    }

    for grp, sub in calib.groupby("group"):
        s = np.clip(sub["n"] / calib["n"].max() * 200, 8, None)
        ax.scatter(
            sub["p_pred"], sub["p_obs"],
            s=s, alpha=0.7, label=grp,
            color=colors.get(grp, "gray"), zorder=2
        )

    ax.set_xlabel("Predicted probability")
    ax.set_ylabel("Observed probability")
    ax.set_xlim(0, max_val)
    ax.set_ylim(0, max_val)

    if title:
        ax.set_title(title, fontsize=10)

    ax.grid(True, alpha=0.3)
    return ax


# ============================================================ #
# GRID: raw cohort calibration, per comb, fixed mode
# ============================================================ #
def plot_cohort_calibration_grid_by_comb(
    df_orig, x_trips, synthetic, combs, mode_idx,
    comb_labels=None, rare_quantile=0.33, tail_quantile=0.33,
    ncols=4, figsize_per_plot=(3.5, 3.5), suptitle=None
):
    n_combs = len(combs)
    if comb_labels is None:
        comb_labels = [f"comb_{i+1}" for i in range(n_combs)]

    nrows = int(np.ceil(n_combs / ncols))
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(figsize_per_plot[0] * ncols, figsize_per_plot[1] * nrows),
        squeeze=False
    )

    for i, comb in enumerate(combs):
        row, col = divmod(i, ncols)
        ax = axes[row][col]

        calib = compute_cohort_calibration_data(
            df_orig, x_trips, synthetic, comb, mode_idx,
            rare_quantile=rare_quantile, tail_quantile=tail_quantile
        )
        plot_cohort_calibration(calib, ax=ax, title=comb_labels[i])

    for i in range(n_combs, nrows * ncols):
        row, col = divmod(i, ncols)
        axes[row][col].axis("off")

    from matplotlib.lines import Line2D

    legend_handles = [
        Line2D([0], [0], marker='o', linestyle='',
            markerfacecolor='#4C72B0', markeredgecolor='none',
            markersize=8, label='normal'),

        Line2D([0], [0], marker='o', linestyle='',
            markerfacecolor='#DD8452', markeredgecolor='none',
            markersize=8, label='rare'),

        Line2D([0], [0], marker='o', linestyle='',
            markerfacecolor='#55A868', markeredgecolor='none',
            markersize=8, label='tail'),

        Line2D([0], [0], marker='o', linestyle='',
            markerfacecolor='#C44E52', markeredgecolor='none',
            markersize=8, label='rare & tail'),
    ]

    fig.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.01),
        ncol=4,
        frameon=False
    )

    if suptitle:
        fig.suptitle(suptitle, fontsize=14)

    plt.tight_layout()
    plt.savefig(path + f"results/plots/cohort_calibration_grid_mode{mode_idx}_{data_split}.png", dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()


    return fig

In [ ]:
# Select random combinations for plotting
comps_to_plot = random.sample(combs, 20)

In [ ]:
fig = plot_cohort_calibration_grid_by_comb(
    df_orig=dfin_disc,
    x_trips=x_trips,
    synthetic=synthetic,
    combs=comps_to_plot,                        # âœ… slice combs as well
    mode_idx=2,
    comb_labels=[str(c) for c in comps_to_plot],
    ncols=4,
)

In [ ]:
def diagnose_rare_tail(df_orig, x_trips, synthetic, comb, mode_idx,
                        rare_quantile=0.25, tail_quantile=0.25):
    if hasattr(x_trips, "numpy"):
        x_trips = x_trips.numpy()
    synthetic_trips = np.array(synthetic["trips"])

    y_true = (x_trips[:, mode_idx] > 0).astype(float)
    y_pred = np.mean(synthetic_trips[:, :, mode_idx] > 0, axis=0)

    df = df_orig[comb].copy()
    df["p_obs"] = y_true
    df["p_pred"] = y_pred

    grouped = (
        df.groupby(comb, observed=False)
        .agg(p_obs=("p_obs", "mean"), p_pred=("p_pred", "mean"), n=("p_obs", "size"))
        .reset_index()
    )
    grouped = grouped[grouped["n"] > 0].copy()

    global_mean = np.average(grouped["p_obs"], weights=grouped["n"])
    grouped["sq_err_model"] = (grouped["p_obs"] - grouped["p_pred"])**2
    grouped["sq_err_baseline"] = (grouped["p_obs"] - global_mean)**2
    grouped["weighted_err_model"] = grouped["sq_err_model"] * grouped["n"]
    grouped["weighted_err_baseline"] = grouped["sq_err_baseline"] * grouped["n"]

    rare_cutoff = grouped["n"].quantile(rare_quantile)
    grouped["is_rare"] = grouped["n"] <= rare_cutoff

    rare = grouped[grouped["is_rare"]].sort_values("weighted_err_model", ascending=False)

    print(f"global_mean (baseline prediction) = {global_mean:.5f}")
    print(f"rare cohorts: {len(rare)} | total n in rare: {rare['n'].sum()}")
    print(f"sum weighted model error (rare)    = {rare['weighted_err_model'].sum():.6f}")
    print(f"sum weighted baseline error (rare)  = {rare['weighted_err_baseline'].sum():.6f}")
    print()

    # âœ… include weighted_err_baseline in the returned columns
    return rare[comb + ["n", "p_obs", "p_pred", "sq_err_model", "sq_err_baseline",
                         "weighted_err_model", "weighted_err_baseline"]]

In [ ]:
for i, c in enumerate(combs):
    result = diagnose_rare_tail(dfin_disc, x_trips, synthetic, c, mode_idx=6, rare_quantile=0.25)
    model_err = result["weighted_err_model"].sum()
    base_err = result["weighted_err_baseline"].sum()
    skill = 1 - model_err/base_err if base_err > 0 else float("nan")
    print(f"comb {i}: {c}")
    print(f"  n_rare={result['n'].sum()}, model_err={model_err:.5f}, base_err={base_err:.5f}, skill_rare={skill:.3f}")
    print()